In [1]:
!pip install transformers accelerate peft torch sentence-transformers faiss-cpu bitsandbytes
!pip install faiss
!apt-get update
!apt-get install -y libstdc++6
!pip install opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 90.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 10.1 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12

In [2]:
import faiss
import numpy as np
import pandas as pd
import pickle
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline ,AutoModelForCausalLM
import torch
from peft import PeftModel ,PeftConfig
from transformers import BitsAndBytesConfig
import os
from accelerate import dispatch_model

In [3]:
#load the customer service intent
with open("/content/drive/MyDrive/Colab Notebooks/chatbot_pipeline/customer_intents.txt", "r") as file:
    customer_intents = set(line.strip().lower() for line in file)

# Load label encoder
with open("/content/drive/MyDrive/Colab Notebooks/chatbot_pipeline/label_encoder.pkl", "rb") as f:
    label_encoder = pickle.load(f)

humen_df=pd.read_csv("/content/drive/MyDrive/Colab Notebooks/chatbot_pipeline/humen_intents.csv")
quant_config = BitsAndBytesConfig(load_in_4bit=True)


/usr/local/lib/python3.11/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.2.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [4]:
device ="cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

#load the sentence transformer model
retrieve_model = SentenceTransformer("/content/drive/MyDrive/Colab Notebooks/chatbot_pipeline/sentence_transformer_model").to(device)
instruction_embeddings = np.load("/content/drive/MyDrive/Colab Notebooks/chatbot_pipeline/instruction_embeddings.npy")
response_embeddings = np.load("/content/drive/MyDrive/Colab Notebooks/chatbot_pipeline/response_embeddings.npy")
faiss_index = faiss.read_index("/content/drive/MyDrive/Colab Notebooks/chatbot_pipeline/instruction_index.faiss")


Using device: cuda


In [5]:
os.environ["WANDB_DISABLED"] = "true"

In [6]:

#load the responses for retrieval
cleaned_texts=pd.read_parquet("/content/drive/MyDrive/Colab Notebooks/chatbot_pipeline/cleaned_texts.parquet")
responses = cleaned_texts["response"].tolist()


#load the intent classification model
intent_model=AutoModelForSequenceClassification.from_pretrained("/content/drive/MyDrive/Colab Notebooks/chatbot_pipeline/intent_classification_model").to(device)
intent_tokenizer=AutoTokenizer.from_pretrained("/content/drive/MyDrive/Colab Notebooks/chatbot_pipeline/intent_classification_model")


In [7]:
base_model_name = "meta-llama/Llama-3.2-1B"
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float32,
    device_map={"": device},
    trust_remote_code=True
)
llama_tokenizer = AutoTokenizer.from_pretrained(base_model_name)
lora_model_path = "/content/drive/MyDrive/Colab Notebooks/chatbot_pipeline/llama_finetuned_model"
config = PeftConfig.from_pretrained(lora_model_path)
llama_model = PeftModel.from_pretrained(base_model, lora_model_path, config=config , is_trainable=True)

#llama_model.print_trainable_parameters()

llama_tokenizer.pad_token = llama_tokenizer.eos_token

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

In [8]:
# ------------------------- INTENT CLASSIFICATION -------------------------


def predict_query_intent(query):
    with torch.no_grad():
        inputs = intent_tokenizer(query, return_tensors="pt", padding=True, truncation=True).to(device)
        outputs = intent_model(**inputs)
        intent_logits = outputs.logits
        intent_index = torch.argmax(intent_logits, dim=1).item()
    predicted_intent = label_encoder.inverse_transform([intent_index])[0]
    #print(f"Predicted Intent from predict_query_intent: {predicted_intent}")
    return predicted_intent





def classify_intent(query):

    predicted_intent = predict_query_intent(query)
    if predicted_intent in customer_intents:
        intent_category = "customer_service"
    elif predicted_intent in humen_df["intents"].tolist():
        intent_category = "human_interaction"
    else:
        intent_category = "unknown"
    #print(f"Classify Intent: {intent_category}")
    return intent_category


In [9]:
# ------------------------- RESPONSE RETRIEVAL -------------------------

def retrieve_response(query):
    query_embedding=retrieve_model.encode(query).reshape(1, -1).astype('float32')
    distance, indicies = faiss_index.search(query_embedding, 1)
    if indicies[0][0] == -1:
        print("No relevant response found in FAISS index!")
        return "I'm not sure how to answer that.", 1.0
    best_response_index = indicies[0][0]
    best_response = cleaned_texts.iloc[best_response_index]["response"]
    #print(f"Retrieved FAISS Response: {best_response} (Distance: {distance[0][0]:.2f})")
    return best_response, distance[0][0]

In [10]:
# ------------------------- RESPONSE GENERATION -------------------------
import re
def generate_response(query, max_length=150):

    """Generates a response using the fine-tuned LLaMA model."""
    formatted_query = f"{query}\nChatBot:"

    inputs = llama_tokenizer(formatted_query , return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
    #print("Input Tensors to Model:", inputs)
    with torch.no_grad():
       output = llama_model.generate(
            **inputs,
            max_length=max_length,
            temperature=0.7,
            top_p=0.8,
            repetition_penalty=1.7,
            pad_token_id=llama_tokenizer.eos_token_id
        )
    response = llama_tokenizer.decode(output[0], skip_special_tokens=True)
    response = response.replace("ChatBot:", "").strip()

    if response.lower().startswith(query.lower()):
        response = response[len(query):].strip()


    response = re.split(r"Intent\s*:", response)[0].strip()
    return response


In [11]:
# ------------------------- CHATBOT PIPELINE -------------------------


def chatbot_pipeline(user_input):
    intent = classify_intent(user_input)
    if intent == "customer_service":
        response, distance = retrieve_response(user_input)
        print(f"Retrieved Response: {response}")
        if distance > 0.6:
            print("Low confidence, generating response...")
            response = generate_response(user_input)
    elif intent== "human_interaction":
        response = generate_response(user_input)
    else:
        response = "I'm not sure how to answer that."

        with open("unknown_queries.txt", "a") as file:
            file.write(user_input + "\n")

        print(f"Logged unknown query: {user_input}")
    return response

In [13]:
# ------------------------- MAIN PROGRAM -------------------------

if __name__ == "__main__":
     while True:
        user_input = input("You: ")
        if user_input.lower() in ["exit", "quit"]:
            print("ChatBot: Goodbye!")
            break
        print(chatbot_pipeline(user_input),"\n")


You: how to cancel order
Retrieved Response: ive realized that youre seeking assistance in canceling an order ill be happy to guide you through the process could you please provide me with the order number or any relevant details about the order you would like to cancel with that information ill be able to provide you with the specific steps to cancel your order successfully
ive realized that youre seeking assistance in canceling an order ill be happy to guide you through the process could you please provide me with the order number or any relevant details about the order you would like to cancel with that information ill be able to provide you with the specific steps to cancel your order successfully 

You: how to place new order
Retrieved Response: thank you for your message and seeking assistance with placing an order were here to help you make this process as smooth and effortless as possible could you please provide us with the details of the items you would like to order once we 